# Imports y Setup

In [8]:
import sys
import os, random, json
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Configuracion Global

In [ ]:
CFG = {
    "data_root": "dataset/BreaKHis_v1/BreaKHis_v1/histology_slides/breast",
    "magnifications": ["40X", "100X", "200X", "400X"],
    "img_size": 224,
    "batch_size": 32,
    "num_epochs": 25,
    "lr": 3e-4,
    "weight_decay": 1e-3,
    "patience": 7,
    "val_split": 0.15,
    "test_split": 0.15,
    "label_smoothing": 0.1,
    "dropout": 0.4,
    "focal_gamma": 2.0,
    "seed": 42,
    "device": "mps" if torch.backends.mps.is_available() else "cpu",
    "save_dir": "outputs/",
}

BINARY_CLASSES = ["benign", "malignant"]
SUBTYPE_CLASSES = [
    "adenosis", "fibroadenoma", "phyllodes_tumor", "tubular_adenoma",
    "ductal_carcinoma", "lobular_carcinoma", "mucinous_carcinoma", "papillary_carcinoma",
]
MAG_CLASSES = ["40X", "100X", "200X", "400X"]

# Parseo del Dataset 

In [ ]:
def parse_dataset(data_root, magnifications=None):
    root = Path(data_root)
    records = []
    mag_filter = set(magnifications) if magnifications else None

    for binary_dir in root.iterdir():
        if not binary_dir.is_dir():
            continue
        try:
            binary_label = BINARY_CLASSES.index(binary_dir.name.lower())
        except ValueError:
            continue

        subtype_candidates = []
        for child in binary_dir.iterdir():
            if not child.is_dir():
                continue
            if child.name.lower().replace(" ", "_") in SUBTYPE_CLASSES:
                subtype_candidates.append(child)
            else:
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        subtype_candidates.append(grandchild)

        for subtype_dir in subtype_candidates:
            subtype_name = subtype_dir.name.lower().replace(" ", "_")
            if subtype_name not in SUBTYPE_CLASSES:
                print(f"  subtipo desconocido: {subtype_name}, se omite")
                continue
            subtype_label = SUBTYPE_CLASSES.index(subtype_name)

            for patient_dir in subtype_dir.iterdir():
                if not patient_dir.is_dir():
                    continue
                patient_id = patient_dir.name

                for mag_dir in patient_dir.iterdir():
                    if not mag_dir.is_dir():
                        continue
                    mag = mag_dir.name
                    if mag_filter and mag not in mag_filter:
                        continue
                    if mag not in MAG_CLASSES:
                        continue
                    mag_label = MAG_CLASSES.index(mag)

                    for img_path in mag_dir.glob("*.png"):
                        records.append({
                            "path": str(img_path),
                            "binary_label": binary_label,
                            "subtype_label": subtype_label,
                            "mag_label": mag_label,
                            "patient_id": patient_id,
                        })

    print(f"\nTotal imagenes encontradas: {len(records)}")
    return records


def split_by_patient(records, val_split=0.15, test_split=0.15, seed=42):
    random.seed(seed)
    patients_by_class = defaultdict(set)
    for r in records:
        patients_by_class[r["binary_label"]].add(r["patient_id"])

    val_patients, test_patients = set(), set()
    for cls, patients in patients_by_class.items():
        patients = list(patients)
        random.shuffle(patients)
        n_val  = max(1, int(len(patients) * val_split))
        n_test = max(1, int(len(patients) * test_split))
        val_patients.update(patients[:n_val])
        test_patients.update(patients[n_val:n_val + n_test])

    train = [r for r in records if r["patient_id"] not in val_patients and r["patient_id"] not in test_patients]
    val   = [r for r in records if r["patient_id"] in val_patients]
    test  = [r for r in records if r["patient_id"] in test_patients]

    print("\nDistribucion por split:")
    for name, split in [("Train", train), ("Val", val), ("Test", test)]:
        counts = defaultdict(int)
        for r in split:
            counts[BINARY_CLASSES[r["binary_label"]]] += 1
        print(f"  {name}: {len(split):5d} imagenes | {dict(counts)}")

    return train, val, test

# PyTorch Dataset & Transforms 

In [ ]:
class BreakHisDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = Image.open(r["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return (
            img,
            torch.tensor(r["binary_label"], dtype=torch.long),
            torch.tensor(r["subtype_label"], dtype=torch.long),
            torch.tensor(r["mag_label"], dtype=torch.long),
        )


def get_transforms(img_size, is_train=True):
    mean = [0.7861, 0.6214, 0.7632]
    std  = [0.1322, 0.1561, 0.1168]

    if is_train:
        return transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(20),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
            transforms.RandomGrayscale(p=0.05),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])


def make_weighted_sampler(records):
    labels = [r["binary_label"] for r in records]
    counts = np.bincount(labels)
    weights = 1.0 / counts
    sample_w = [weights[l] for l in labels]
    return WeightedRandomSampler(sample_w, len(sample_w), replacement=True)

# Modelo

In [ ]:
class EfficientNetMultiTask(nn.Module):
    def __init__(self, n_binary=2, n_subtype=8, n_mag=4, dropout=0.4):
        super().__init__()
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = backbone.features
        self.avgpool  = backbone.avgpool
        feat_dim = backbone.classifier[1].in_features

        self.binary_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 256),
            nn.SiLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, n_binary),
        )
        self.subtype_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.SiLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout * 0.5),
            nn.Linear(512, n_subtype),
        )
        self.mag_head = nn.Sequential(
            nn.Dropout(dropout * 0.5),
            nn.Linear(feat_dim, 128),
            nn.SiLU(),
            nn.Linear(128, n_mag),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.binary_head(x), self.subtype_head(x), self.mag_head(x)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs, targets,
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction="none",
        )
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()


def compute_class_weights(records, n_classes, label_key):
    counts = np.zeros(n_classes)
    for r in records:
        counts[r[label_key]] += 1
    weights = 1.0 / (counts + 1e-6)
    weights /= weights.sum()
    return torch.tensor(weights, dtype=torch.float)


def train_one_epoch(model, loader, optimizer, criterion_b, criterion_s, criterion_m, device, w=(0.6, 0.3, 0.1)):
    model.train()
    total_loss, correct_b, n = 0.0, 0, 0

    for imgs, y_b, y_s, y_m in loader:
        imgs, y_b, y_s, y_m = imgs.to(device), y_b.to(device), y_s.to(device), y_m.to(device)
        out_b, out_s, out_m = model(imgs)

        loss = (
            w[0] * criterion_b(out_b, y_b)
            + w[1] * criterion_s(out_s, y_s)
            + w[2] * criterion_m(out_m, y_m)
        )

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        correct_b  += (out_b.argmax(1) == y_b).sum().item()
        n += imgs.size(0)

    return total_loss / n, correct_b / n


@torch.no_grad()
def evaluate(model, loader, criterion_b, criterion_s, criterion_m, device):
    model.eval()
    total_loss = 0.0
    all_y, all_pred, all_prob, all_mag = [], [], [], []
    n = 0

    for imgs, y_b, y_s, y_m in loader:
        imgs, y_b, y_s, y_m = imgs.to(device), y_b.to(device), y_s.to(device), y_m.to(device)
        out_b, out_s, out_m = model(imgs)

        loss = (
            0.6 * criterion_b(out_b, y_b)
            + 0.3 * criterion_s(out_s, y_s)
            + 0.1 * criterion_m(out_m, y_m)
        )

        total_loss += loss.item() * imgs.size(0)
        probs = torch.softmax(out_b, dim=1)[:, 1]
        all_y.extend(y_b.cpu().numpy())
        all_pred.extend(out_b.argmax(1).cpu().numpy())
        all_prob.extend(probs.cpu().numpy())
        all_mag.extend(y_m.cpu().numpy())
        n += imgs.size(0)

    acc = sum(p == y for p, y in zip(all_pred, all_y)) / n
    try:
        auc = roc_auc_score(all_y, all_prob)
    except Exception:
        auc = 0.0
    f1 = f1_score(all_y, all_pred, average="macro", zero_division=0)

    return total_loss / n, acc, auc, f1, all_y, all_pred, all_prob, all_mag


def unfreeze_schedule(model, epoch):
    if epoch == 0:
        for p in model.features.parameters():
            p.requires_grad = False
        print("  Backbone congelado")
    elif epoch == 5:
        for i, block in enumerate(model.features):
            if i >= 6:
                for p in block.parameters():
                    p.requires_grad = True
        print("  Bloques 6-7 desbloqueados")
    elif epoch == 10:
        for p in model.features.parameters():
            p.requires_grad = True
        print("  Backbone completo desbloqueado")


def plot_class_distribution(records_dict, save_dir):
    n = len(records_dict)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]
    colors = ["#4C8BE2", "#E25F5F"]
    for ax, (split_name, records) in zip(axes, records_dict.items()):
        counts = defaultdict(int)
        for r in records:
            counts[BINARY_CLASSES[r["binary_label"]]] += 1
        vals = [counts.get(c, 0) for c in BINARY_CLASSES]
        bars = ax.bar(BINARY_CLASSES, vals, color=colors)
        ax.bar_label(bars, padding=3)
        ax.set_title(f"Distribucion — {split_name}")
        ax.set_ylabel("Numero de imagenes")
        ax.set_ylim(0, max(vals) * 1.15)
        ax.grid(axis="y", alpha=0.3)
    plt.suptitle("Distribucion de clases por conjunto", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(Path(save_dir) / "class_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("  class_distribution.png guardado")


def plot_history(history, save_dir):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    epochs = range(1, len(history["train_loss"]) + 1)

    axes[0, 0].plot(epochs, history["train_loss"], label="Train", color="#4C8BE2")
    axes[0, 0].plot(epochs, history["val_loss"],   label="Val",   color="#E25F5F")
    axes[0, 0].set_title("Funcion de Perdida (Focal Loss)")
    axes[0, 0].set_xlabel("Epoca"); axes[0, 0].set_ylabel("Loss")
    axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(epochs, history["train_acc"], label="Train", color="#4C8BE2")
    axes[0, 1].plot(epochs, history["val_acc"],   label="Val",   color="#E25F5F")
    axes[0, 1].set_title("Accuracy Binaria (Benigno/Maligno)")
    axes[0, 1].set_xlabel("Epoca"); axes[0, 1].set_ylabel("Accuracy")
    axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(epochs, history["val_auc"], color="#2ECC71", label="Val AUC-ROC")
    axes[1, 0].axhline(y=0.5, linestyle="--", color="gray", alpha=0.5, label="Azar")
    axes[1, 0].set_title("AUC-ROC en Validacion")
    axes[1, 0].set_xlabel("Epoca"); axes[1, 0].set_ylabel("AUC")
    axes[1, 0].set_ylim([0.4, 1.0]); axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(epochs, history["val_f1"], color="#9B59B6", label="Val F1 Macro")
    axes[1, 1].set_title("F1-Score Macro en Validacion")
    axes[1, 1].set_xlabel("Epoca"); axes[1, 1].set_ylabel("F1")
    axes[1, 1].set_ylim([0, 1.05]); axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

    plt.suptitle("Historial de Entrenamiento — EfficientNet-B0", fontsize=14)
    plt.tight_layout()
    plt.savefig(Path(save_dir) / "training_history.png", dpi=150)
    plt.show()
    print("  training_history.png guardado")


def plot_lr_schedule(history, save_dir):
    if not history.get("lr"):
        return
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(range(1, len(history["lr"]) + 1), history["lr"], marker="o", markersize=3, color="#E67E22")
    ax.set_xlabel("Epoca"); ax.set_ylabel("Learning Rate")
    ax.set_title("Evolucion del Learning Rate (CosineAnnealing)")
    ax.set_yscale("log"); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(save_dir) / "lr_schedule.png", dpi=150)
    plt.show()
    print("  lr_schedule.png guardado")


def plot_confusion(y_true, y_pred, save_dir, suffix="val"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, normalize, title in zip(axes, [None, "true"], ["Conteos absolutos", "Normalizada por clase real"]):
        cm = confusion_matrix(y_true, y_pred, normalize=normalize)
        fmt = ".2f" if normalize else "d"
        sns.heatmap(cm, annot=True, fmt=fmt, cmap="Blues",
                    xticklabels=BINARY_CLASSES, yticklabels=BINARY_CLASSES, ax=ax, linewidths=0.5)
        ax.set_ylabel("Real"); ax.set_xlabel("Predicho")
        ax.set_title(f"Matriz de Confusion — {title}")
    plt.suptitle(f"Benigno vs Maligno ({suffix.upper()})", fontsize=13)
    plt.tight_layout()
    plt.savefig(Path(save_dir) / f"confusion_matrix_{suffix}.png", dpi=150)
    plt.show()
    print(f"  confusion_matrix_{suffix}.png guardado")


def plot_roc_curve(y_true, y_prob, save_dir, suffix="val"):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC (AUC = {auc:.4f})")
    ax.plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--", label="Clasificador aleatorio")
    ax.fill_between(fpr, tpr, alpha=0.1, color="darkorange")
    ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("Tasa de Falsos Positivos (1 - Especificidad)")
    ax.set_ylabel("Tasa de Verdaderos Positivos (Sensibilidad)")
    ax.set_title(f"Curva ROC — Benigno vs Maligno ({suffix.upper()})")
    ax.legend(loc="lower right"); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(save_dir) / f"roc_curve_{suffix}.png", dpi=150)
    plt.show()
    print(f"  roc_curve_{suffix}.png guardado")


def plot_pr_curve(y_true, y_prob, save_dir, suffix="val"):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    baseline = sum(y_true) / len(y_true)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(recall, precision, color="#27AE60", lw=2, label=f"PR curve (AP = {ap:.4f})")
    ax.axhline(y=baseline, linestyle="--", color="gray", alpha=0.6,
               label=f"Baseline (proporcion maligno = {baseline:.2f})")
    ax.fill_between(recall, precision, alpha=0.1, color="#27AE60")
    ax.set_xlabel("Recall (Sensibilidad)"); ax.set_ylabel("Precision")
    ax.set_title(f"Curva Precision-Recall — Maligno ({suffix.upper()})")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(save_dir) / f"pr_curve_{suffix}.png", dpi=150)
    plt.show()
    print(f"  pr_curve_{suffix}.png guardado")


def plot_metrics_per_class(y_true, y_pred, save_dir, suffix="val"):
    metrics = {}
    for cls_idx, cls_name in enumerate(BINARY_CLASSES):
        y_b_true = [1 if y == cls_idx else 0 for y in y_true]
        y_b_pred = [1 if y == cls_idx else 0 for y in y_pred]
        metrics[cls_name] = {
            "Precision": precision_score(y_b_true, y_b_pred, zero_division=0),
            "Recall":    recall_score(y_b_true, y_b_pred, zero_division=0),
            "F1-Score":  f1_score(y_b_true, y_b_pred, zero_division=0),
        }
    x = np.arange(len(BINARY_CLASSES))
    width = 0.25
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, (metric, color) in enumerate(zip(["Precision", "Recall", "F1-Score"], ["#3498DB", "#E74C3C", "#2ECC71"])):
        vals = [metrics[cls][metric] for cls in BINARY_CLASSES]
        bars = ax.bar(x + i * width, vals, width, label=metric, color=color, alpha=0.85)
        ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=8)
    ax.set_xticks(x + width); ax.set_xticklabels(BINARY_CLASSES, fontsize=11)
    ax.set_ylim([0, 1.15]); ax.set_ylabel("Score")
    ax.set_title(f"Metricas por Clase — Benigno vs Maligno ({suffix.upper()})")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(save_dir) / f"metrics_per_class_{suffix}.png", dpi=150)
    plt.show()
    print(f"  metrics_per_class_{suffix}.png guardado")


def plot_accuracy_by_magnification(y_true, y_pred, all_mag, save_dir, suffix="val"):
    mag_data = {}
    for mag_idx, mag_name in enumerate(MAG_CLASSES):
        indices = [i for i, m in enumerate(all_mag) if m == mag_idx]
        if not indices:
            continue
        correct = sum(y_pred[i] == y_true[i] for i in indices)
        mag_data[mag_name] = {"acc": correct / len(indices), "n": len(indices)}
    fig, ax = plt.subplots(figsize=(7, 4))
    names = list(mag_data.keys())
    accs  = [mag_data[m]["acc"] for m in names]
    ns    = [mag_data[m]["n"]   for m in names]
    bars = ax.bar(names, accs, color="#1ABC9C", alpha=0.85, edgecolor="white")
    for bar, acc, n in zip(bars, accs, ns):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{acc:.3f}\n(n={n})", ha="center", va="bottom", fontsize=9)
    ax.set_ylim([0, 1.15]); ax.set_ylabel("Accuracy")
    ax.set_xlabel("Magnificacion")
    ax.set_title(f"Accuracy por Magnificacion ({suffix.upper()})")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(save_dir) / f"accuracy_by_magnification_{suffix}.png", dpi=150)
    plt.show()
    print(f"  accuracy_by_magnification_{suffix}.png guardado")


def plot_summary_report(results, save_dir):
    metrics_names = ["Accuracy", "AUC-ROC", "F1-Macro"]
    val_vals  = [results["val"]["acc"],  results["val"]["auc"],  results["val"]["f1"]]
    test_vals = [results["test"]["acc"], results["test"]["auc"], results["test"]["f1"]]
    x = np.arange(len(metrics_names))
    width = 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    b1 = ax.bar(x - width / 2, val_vals,  width, label="Validacion", color="#3498DB", alpha=0.85)
    b2 = ax.bar(x + width / 2, test_vals, width, label="Test",       color="#E74C3C", alpha=0.85)
    ax.bar_label(b1, fmt="%.4f", padding=3, fontsize=9)
    ax.bar_label(b2, fmt="%.4f", padding=3, fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(metrics_names, fontsize=11)
    ax.set_ylim([0, 1.15]); ax.set_ylabel("Score")
    ax.set_title("Resumen de Metricas — Validacion vs Test")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(save_dir) / "summary_val_vs_test.png", dpi=150)
    plt.show()
    print("  summary_val_vs_test.png guardado")

# Pipeline principal

In [ ]:
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
random.seed(CFG["seed"])

os.makedirs(CFG["save_dir"], exist_ok=True)
device = torch.device(CFG["device"])
print(f"\nDispositivo: {device}")

print("\nLeyendo dataset...")
records = parse_dataset(CFG["data_root"], CFG["magnifications"])
train_records, val_records, test_records = split_by_patient(
    records, CFG["val_split"], CFG["test_split"], CFG["seed"]
)

plot_class_distribution(
    {"Train": train_records, "Validacion": val_records, "Test": test_records},
    CFG["save_dir"],
)

train_ds = BreakHisDataset(train_records, transform=get_transforms(CFG["img_size"], is_train=True))
val_ds   = BreakHisDataset(val_records,   transform=get_transforms(CFG["img_size"], is_train=False))
test_ds  = BreakHisDataset(test_records,  transform=get_transforms(CFG["img_size"], is_train=False))

sampler = make_weighted_sampler(train_records)
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], sampler=sampler,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False,    num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"], shuffle=False,    num_workers=0, pin_memory=False)

print("\nInicializando EfficientNet-B0...")
model = EfficientNetMultiTask(dropout=CFG["dropout"]).to(device)

w_binary  = compute_class_weights(train_records, 2, "binary_label").to(device)
w_subtype = compute_class_weights(train_records, 8, "subtype_label").to(device)
print(f"  Pesos binarios: benigno={w_binary[0]:.4f}, maligno={w_binary[1]:.4f}")

criterion_b = FocalLoss(gamma=CFG["focal_gamma"], weight=w_binary,  label_smoothing=CFG["label_smoothing"])
criterion_s = FocalLoss(gamma=CFG["focal_gamma"], weight=w_subtype, label_smoothing=CFG["label_smoothing"])
criterion_m = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG["num_epochs"], eta_min=1e-6
)

best_auc = 0.0
patience_count = 0
history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": [], "val_auc": [], "val_f1": [],
    "lr": [],
}

print(f"\nEntrenando {CFG['num_epochs']} epocas...\n")

for epoch in range(CFG["num_epochs"]):
    unfreeze_schedule(model, epoch)

    if epoch in (5, 10):
        lr = CFG["lr"] * 0.1 if epoch == 5 else CFG["lr"] * 0.01
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr, weight_decay=CFG["weight_decay"],
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG["num_epochs"] - epoch, eta_min=1e-7
        )

    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion_b, criterion_s, criterion_m, device
    )
    val_loss, val_acc, val_auc, val_f1, y_true, y_pred, y_prob, y_mag = evaluate(
        model, val_loader, criterion_b, criterion_s, criterion_m, device
    )
    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_auc"].append(val_auc)
    history["val_f1"].append(val_f1)
    history["lr"].append(current_lr)

    print(
        f"Epoca {epoch+1:02d}/{CFG['num_epochs']}  "
        f"| Train Loss: {train_loss:.4f}  Acc: {train_acc:.3f}"
        f"  | Val Loss: {val_loss:.4f}  Acc: {val_acc:.3f}  AUC: {val_auc:.3f}  F1: {val_f1:.3f}"
        f"  | LR: {current_lr:.2e}"
    )

    if val_auc > best_auc:
        best_auc = val_auc
        patience_count = 0
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "val_auc": float(val_auc),
            "val_acc": float(val_acc),
            "val_f1":  float(val_f1),
            "cfg": {k: v for k, v in CFG.items() if isinstance(v, (str, int, float, list, bool, type(None)))},
        }, Path(CFG["save_dir"]) / "best_model.pt")
        print(f"  Mejor modelo guardado (AUC: {val_auc:.4f})")
    else:
        patience_count += 1
        if patience_count >= CFG["patience"]:
            print(f"\nEarly stopping en epoca {epoch+1}")
            break

plot_history(history, CFG["save_dir"])
plot_lr_schedule(history, CFG["save_dir"])

with open(Path(CFG["save_dir"]) / "history.json", "w") as f:
    json.dump(history, f, indent=2)

ckpt = torch.load(
    Path(CFG["save_dir"]) / "best_model.pt",
    map_location=device,
    weights_only=False,
)
model.load_state_dict(ckpt["model_state"])

results = {}
for split_name, loader in [("val", val_loader), ("test", test_loader)]:
    print(f"\n{'='*55}")
    print(f"  EVALUACION FINAL — {split_name.upper()}")
    print(f"{'='*55}")
    _, acc, auc, f1_mac, y_true, y_pred, y_prob, y_mag = evaluate(
        model, loader, criterion_b, criterion_s, criterion_m, device
    )
    print(classification_report(y_true, y_pred, target_names=BINARY_CLASSES, digits=4))
    print(f"  AUC-ROC: {auc:.4f}  |  F1 Macro: {f1_mac:.4f}  |  Accuracy: {acc:.4f}")

    results[split_name] = {"acc": acc, "auc": auc, "f1": f1_mac}

    plot_confusion(y_true, y_pred, CFG["save_dir"], suffix=split_name)
    plot_roc_curve(y_true, y_prob, CFG["save_dir"], suffix=split_name)
    plot_pr_curve(y_true, y_prob, CFG["save_dir"], suffix=split_name)
    plot_metrics_per_class(y_true, y_pred, CFG["save_dir"], suffix=split_name)
    plot_accuracy_by_magnification(y_true, y_pred, y_mag, CFG["save_dir"], suffix=split_name)

plot_summary_report(results, CFG["save_dir"])

print(f"\nEntrenamiento completado.")
print(f"   Mejor AUC-ROC (val): {best_auc:.4f}")
print(f"   Graficas guardadas en: {CFG['save_dir']}")
print("\nGraficas generadas:")
for f in sorted(Path(CFG["save_dir"]).glob("*.png")):
    print(f"   {f.name}")